# Weakly-Supervised Bone Segmentation — FracAtlas
### Pipeline: DenseNet121 → LayerCAM → SAM ViT-B → U-Net

**Stages:**
1. Setup & Dataset Download
2. Stage 1 — Anatomy Classifier Training
3. Stage 2 — Pseudo Mask Generation (LayerCAM + SAM)
4. Stage 3 — U-Net Segmentation Training
5. Inference & Visualization

## 1. Setup

In [ ]:
# Mount Google Drive — outputs will be saved here across sessions
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone repo (skip if already cloned)
import os
if not os.path.exists('/content/Thesis'):
    !git clone https://github.com/itsthang333/Thesis.git /content/Thesis
else:
    print('Repo already exists, pulling latest...')
    !git -C /content/Thesis pull

%cd /content/Thesis/project
!pwd

In [ ]:
# Install dependencies
!pip install -q torch torchvision numpy pillow tqdm opencv-python kagglehub
!pip install -q git+https://github.com/facebookresearch/segment-anything.git
print('All dependencies installed.')

In [ ]:
# Download FracAtlas dataset from Kaggle
import kagglehub

DATASET_PATH = kagglehub.dataset_download('mahmudulhasantasin/fracatlas-original-dataset')
print(f'Dataset path: {DATASET_PATH}')

# Verify structure
import os
for root, dirs, files in os.walk(DATASET_PATH):
    level = root.replace(DATASET_PATH, '').count(os.sep)
    if level < 2:
        print('  ' * level + os.path.basename(root) + '/')
    if level == 1:
        print('  ' * (level + 1) + f'({len(files)} files)')

In [ ]:
# Download SAM ViT-B checkpoint (~375 MB)
# Saved to Drive so it persists across Colab sessions
import os, urllib.request

SAM_CHECKPOINT_DIR = '/content/drive/MyDrive/ThesisOutputs/checkpoints'
SAM_CHECKPOINT_PATH = os.path.join(SAM_CHECKPOINT_DIR, 'sam_vit_b_01ec64.pth')
SAM_DOWNLOAD_URL = 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth'

os.makedirs(SAM_CHECKPOINT_DIR, exist_ok=True)

if os.path.exists(SAM_CHECKPOINT_PATH):
    print(f'SAM checkpoint already exists: {SAM_CHECKPOINT_PATH}')
else:
    print('Downloading SAM ViT-B checkpoint (~375 MB)...')
    urllib.request.urlretrieve(SAM_DOWNLOAD_URL, SAM_CHECKPOINT_PATH)
    print(f'Saved to {SAM_CHECKPOINT_PATH}')

## 2. Stage 1 — Anatomy Classifier Training

Train DenseNet121 with BCEWithLogitsLoss on 4 anatomy labels: `hand, leg, hip, shoulder`.

In [ ]:
CLASSIFIER_OUTPUT = '/content/drive/MyDrive/ThesisOutputs/classifier'

!python train_classifier.py \
  --data-root "{DATASET_PATH}" \
  --target-columns hand,leg,hip,shoulder \
  --image-size 384 \
  --batch-size 4 \
  --num-workers 2 \
  --epochs 25 \
  --output-dir "{CLASSIFIER_OUTPUT}"

## 3. Stage 2 — Pseudo Mask Generation (LayerCAM + SAM)

Pipeline:
- **LayerCAM** on denseblock2/3/4 → confidence-filtered weighted fusion
- **Adaptive threshold** (percentile 85) → peak prompts
- **SAM ViT-B** generates candidate masks from prompts
- **CAM-guided selection** (mean CAM score ≥ 0.4) + logical-OR fusion
- **Morphological refinement** (closing → opening → fill_holes → remove_small)

In [ ]:
PSEUDO_MASK_OUTPUT = '/content/drive/MyDrive/ThesisOutputs/pseudo_masks'
CLASSIFIER_CHECKPOINT = f'{CLASSIFIER_OUTPUT}/best_classifier.pt'

!python generate_pseudo_masks.py \
  --data-root "{DATASET_PATH}" \
  --classifier-checkpoint "{CLASSIFIER_CHECKPOINT}" \
  --sam-checkpoint "{SAM_CHECKPOINT_PATH}" \
  --target-columns hand,leg,hip,shoulder \
  --image-size 384 \
  --batch-size 1 \
  --num-workers 2 \
  --confidence-threshold 0.5 \
  --cam-percentile 85.0 \
  --max-points 5 \
  --min-component-area 100 \
  --mask-score-threshold 0.4 \
  --closing-kernel 5 \
  --opening-kernel 3 \
  --min-size 200 \
  --output-dir "{PSEUDO_MASK_OUTPUT}"

In [ ]:
# Sanity check: count generated masks and show a sample
import glob, os
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

mask_files = glob.glob(f'{PSEUDO_MASK_OUTPUT}/masks/*.png')
print(f'Generated {len(mask_files)} pseudo masks')

if mask_files:
    sample_path = mask_files[0]
    stem = os.path.splitext(os.path.basename(sample_path))[0]
    overlay_path = f'{PSEUDO_MASK_OUTPUT}/overlays/{stem}_fused_layercam.png'

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(np.array(Image.open(sample_path)), cmap='gray')
    axes[0].set_title('Pseudo Mask')
    axes[0].axis('off')

    if os.path.exists(overlay_path):
        axes[1].imshow(np.array(Image.open(overlay_path)))
        axes[1].set_title('Fused LayerCAM Overlay')
        axes[1].axis('off')

    plt.suptitle(f'Sample: {stem}', fontsize=11)
    plt.tight_layout()
    plt.show()

## 4. Stage 3 — U-Net Segmentation Training

Train U-Net (encoder 64→1024) with `0.5 × BCE + 0.5 × Dice` loss on the pseudo masks.

In [ ]:
SEGMENTATION_OUTPUT = '/content/drive/MyDrive/ThesisOutputs/segmentation'

!python train_segmentation.py \
  --data-root "{DATASET_PATH}" \
  --mask-root "{PSEUDO_MASK_OUTPUT}/masks" \
  --image-size 384 \
  --batch-size 4 \
  --num-workers 2 \
  --epochs 25 \
  --output-dir "{SEGMENTATION_OUTPUT}"

In [ ]:
# Plot training curves
import pandas as pd
import matplotlib.pyplot as plt
import os

log_path = f'{SEGMENTATION_OUTPUT}/training_log.csv'
if os.path.exists(log_path):
    df = pd.read_csv(log_path)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    for ax, metric in zip(axes, ['loss', 'dice', 'iou']):
        ax.plot(df['epoch'], df[f'train_{metric}'], label='train')
        ax.plot(df['epoch'], df[f'val_{metric}'], label='val')
        ax.set_title(metric.capitalize())
        ax.set_xlabel('Epoch')
        ax.legend()

    plt.suptitle('U-Net Training Curves', fontsize=13)
    plt.tight_layout()
    plt.show()

    best = df.loc[df['val_dice'].idxmax()]
    print(f"Best epoch: {int(best['epoch'])} | "
          f"Val Dice: {best['val_dice']:.4f} | "
          f"Val IoU:  {best['val_iou']:.4f}")

## 5. Inference & Visualization

In [ ]:
# Pick a random test image from the dataset
import glob, random, os

all_images = (
    glob.glob(f'{DATASET_PATH}/**/*.jpg', recursive=True) +
    glob.glob(f'{DATASET_PATH}/**/*.jpeg', recursive=True) +
    glob.glob(f'{DATASET_PATH}/**/*.png', recursive=True)
)
TEST_IMAGE = random.choice(all_images)
print(f'Test image: {TEST_IMAGE}')

INFERENCE_OUTPUT = '/content/drive/MyDrive/ThesisOutputs/inference'

!python inference.py \
  --image-path "{TEST_IMAGE}" \
  --classifier-checkpoint "{CLASSIFIER_CHECKPOINT}" \
  --segmentation-checkpoint "{SEGMENTATION_OUTPUT}/best_unet.pt" \
  --sam-checkpoint "{SAM_CHECKPOINT_PATH}" \
  --image-size 384 \
  --confidence-threshold 0.5 \
  --cam-percentile 85.0 \
  --max-points 5 \
  --min-component-area 100 \
  --mask-score-threshold 0.4 \
  --closing-kernel 5 \
  --opening-kernel 3 \
  --min-size 200 \
  --output-dir "{INFERENCE_OUTPUT}"

In [ ]:
# Visualize full pipeline output
import os
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

stem = os.path.splitext(os.path.basename(TEST_IMAGE))[0]
panels = {
    'Original':           TEST_IMAGE,
    'Fused LayerCAM':     f'{INFERENCE_OUTPUT}/{stem}_fused_layercam.png',
    'Pseudo Mask (SAM)':  f'{INFERENCE_OUTPUT}/{stem}_pseudo_mask.png',
    'Segmentation Mask':  f'{INFERENCE_OUTPUT}/{stem}_segmentation_mask.png',
    'Final Overlay':      f'{INFERENCE_OUTPUT}/{stem}_final_overlay.png',
}

missing = [title for title, path in panels.items() if not os.path.exists(path)]
if missing:
    print(f'WARNING: missing output files: {missing}')

fig, axes = plt.subplots(1, len(panels), figsize=(20, 4))
for ax, (title, path) in zip(axes, panels.items()):
    if os.path.exists(path):
        ax.imshow(np.array(Image.open(path).convert('RGB')))
    else:
        ax.text(0.5, 0.5, 'Not found', ha='center', va='center', transform=ax.transAxes)
    ax.set_title(title, fontsize=9)
    ax.axis('off')

plt.suptitle(f'Pipeline Output — {stem}', fontsize=11)
plt.tight_layout()
plt.show()